# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiknaxTheGreek/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Signal checks + rule reasoning

### Signal 1 — content age / staleness

**Hypothesis:** older content may be a stronger candidate for refresh. This is the required flag-linked signal check because staleness sits behind FlyRank refresh logic.

The feature itself was already selected in Assignment 4 without using April. Here, April is used only as the future outcome for evaluating that pre-selected March-safe signal.

Age buckets are fixed in advance for interpretability: **0–90**, **91–180**, **181–365**, and **366+ days**. The executed table prints `n` for every bucket.

**Verdict: MIXED**

The broad direction supports a staleness effect, but not cleanly enough to call it confirmed. The youngest pages have the lowest observed decline rate (**55.72%**) and the oldest pages the highest (**84.96%**), while the two middle buckets reverse order (**72.08%** for 91–180 days versus **65.81%** for 181–365 days). Median future impression change is also most negative for the 366+ day bucket. Staleness is therefore informative directionally, but a simple monotonic 'older always means worse' assumption is not supported by these buckets.

### Signal 2 — CTR relative to search position

**Hypothesis:** pages with unusually weak CTR for their March search position may be stronger candidates for CTR/content-fix review.

This follows the FlyRank CTR-fix idea while staying inside the Assignment 4 contract. Both `aggregate_ctr` and `median_position` are March-only features already selected before April was examined.

Pages are first placed into fixed March position bands: **1–3**, **4–10**, **11–20**, and **21+**. The first attempted normalization divided each page's CTR by its position-band median, but the executed March data showed at least one band has a median CTR of zero, making that ratio invalid. The audit therefore uses a safer March-only relative measure: each page's **CTR percentile rank among pages in the same position band**. Those ranks are bucketed using fixed quartiles: **bottom 25%**, **25–50%**, **50–75%**, and **top 25%**. April is still used only to evaluate the pre-defined signal.

**Verdict: CONFIRMED**

The result is directionally ordered across all four March-only CTR-relative-to-position buckets. Observed future decline falls from **74.66%** in the bottom quartile to **71.14%**, **65.46%**, and **52.33%** in the top quartile. Median future impression change also improves monotonically from **−34.70%** to **−30.12%**, **−25.30%**, and **−4.71%**. Mean future change moves in the same direction and becomes positive in the upper half. This supports the hypothesis that unusually weak CTR relative to comparable ranking position is a useful review signal.

The March diagnostic also explains why percentile ranking is preferable to a CTR/median ratio here: the **21+ position band has 63.75% zero-CTR pages and a median CTR of zero**, so a ratio to the median would be undefined. The within-band percentile construction handles that observed data shape without using April to tune the signal.

With both signal verdicts now locked — **staleness: MIXED** and **CTR relative to position: CONFIRMED** — the next step is to encode one transparent baseline rule.


In [1]:
# STEP 1A — reconstruct the locked Assignment-4 POC and audit staleness.
# This cell intentionally uses the exact same population rules as w03_data_contract.ipynb.

import os
import duckdb
import numpy as np
import pandas as pd

# Hugging Face token stays private: read from environment / Colab Secrets only.
HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass

if not HF_TOKEN:
    raise RuntimeError(
        "HF_TOKEN is missing. Add a Hugging Face READ token as a Colab Secret named HF_TOKEN."
    )

con = duckdb.connect()
safe_token = HF_TOKEN.replace("'", "''")
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{safe_token}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
MARCH = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"
APRIL = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-04/*.parquet')"
DIM_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"

# 1) Exact longitudinal eligibility from Assignment 4: >=20 usable GSC days in both months.
march_cov = con.sql(f"""
    SELECT client_hash_id, content_hash_id,
           COUNT(DISTINCT report_date) AS march_usable_days
    FROM {MARCH}
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
""").df()

april_cov = con.sql(f"""
    SELECT client_hash_id, content_hash_id,
           COUNT(DISTINCT report_date) AS april_usable_days
    FROM {APRIL}
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
""").df()

matched_keys = (
    march_cov[march_cov["march_usable_days"] >= 20]
    .merge(april_cov, on=["client_hash_id", "content_hash_id"], how="inner")
)
matched_keys = matched_keys[matched_keys["april_usable_days"] >= 20][
    ["client_hash_id", "content_hash_id"]
].drop_duplicates()
con.register("matched_keys", matched_keys)

# 2) Reapply the locked March exposure strata used only for balanced sampling.
march_exposure = con.sql(f"""
    SELECT
        f.client_hash_id,
        f.content_hash_id,
        SUM(f.gsc_impressions)::DOUBLE / COUNT(DISTINCT f.report_date)
            AS march_avg_impressions_per_day
    FROM {MARCH} AS f
    INNER JOIN matched_keys AS k USING (client_hash_id, content_hash_id)
    WHERE f.gsc_data_available IS TRUE
    GROUP BY f.client_hash_id, f.content_hash_id
""").df()

march_exposure["exposure_tier"] = pd.cut(
    march_exposure["march_avg_impressions_per_day"],
    bins=[-float("inf"), 13.42, 58.25, float("inf")],
    labels=["Low", "Medium", "High"],
    include_lowest=True,
)

client_tier_counts = (
    march_exposure.groupby(["client_hash_id", "exposure_tier"], observed=False)
    .size().unstack(fill_value=0)
    .reindex(columns=["Low", "Medium", "High"], fill_value=0)
)
eligible_clients = client_tier_counts[
    client_tier_counts.min(axis=1) >= 40
].index.tolist()

balanced_poc = (
    march_exposure[march_exposure["client_hash_id"].isin(eligible_clients)]
    .sort_values(["client_hash_id", "exposure_tier", "content_hash_id"])
    .groupby(["client_hash_id", "exposure_tier"], observed=False, group_keys=False)
    .head(40)
    .reset_index(drop=True)
)

# 3) March-safe staleness feature: page age as known on 31 March 2026.
balanced_keys = balanced_poc[["client_hash_id", "content_hash_id"]].drop_duplicates()
con.register("balanced_keys", balanced_keys)

age_frame = con.sql(f"""
    SELECT
        d.client_hash_id,
        d.content_hash_id,
        DATE_DIFF('day', d.content_created_date, DATE '2026-03-31')::DOUBLE
            AS content_age_days
    FROM {DIM_CONTENT} AS d
    INNER JOIN balanced_keys AS k USING (client_hash_id, content_hash_id)
""").df()

# 4) Future outcome only: March -> April relative change in average impressions per usable day.
march_target = con.sql(f"""
    SELECT f.client_hash_id, f.content_hash_id,
           SUM(f.gsc_impressions)::DOUBLE / COUNT(DISTINCT f.report_date)
               AS march_avg_impressions_per_day
    FROM {MARCH} AS f
    INNER JOIN balanced_keys AS k USING (client_hash_id, content_hash_id)
    WHERE f.gsc_data_available IS TRUE
    GROUP BY f.client_hash_id, f.content_hash_id
""").df()

april_target = con.sql(f"""
    SELECT f.client_hash_id, f.content_hash_id,
           SUM(f.gsc_impressions)::DOUBLE / COUNT(DISTINCT f.report_date)
               AS april_avg_impressions_per_day
    FROM {APRIL} AS f
    INNER JOIN balanced_keys AS k USING (client_hash_id, content_hash_id)
    WHERE f.gsc_data_available IS TRUE
    GROUP BY f.client_hash_id, f.content_hash_id
""").df()

signal_frame = (
    age_frame
    .merge(march_target, on=["client_hash_id", "content_hash_id"], how="inner")
    .merge(april_target, on=["client_hash_id", "content_hash_id"], how="inner")
)
signal_frame["future_impression_change"] = (
    signal_frame["april_avg_impressions_per_day"]
    - signal_frame["march_avg_impressions_per_day"]
) / signal_frame["march_avg_impressions_per_day"]
signal_frame["future_decline"] = signal_frame["future_impression_change"] < 0

signal_frame["age_bucket"] = pd.cut(
    signal_frame["content_age_days"],
    bins=[-float("inf"), 90, 180, 365, float("inf")],
    labels=["0-90 days", "91-180 days", "181-365 days", "366+ days"],
)

staleness_table = (
    signal_frame.groupby("age_bucket", observed=False)
    .agg(
        n=("content_hash_id", "size"),
        decline_rate=("future_decline", "mean"),
        median_future_impression_change=("future_impression_change", "median"),
        mean_future_impression_change=("future_impression_change", "mean"),
    )
    .reset_index()
)
staleness_table["decline_rate_pct"] = 100 * staleness_table.pop("decline_rate")

print("Locked POC clients:", balanced_poc["client_hash_id"].nunique())
print("Locked POC pages:", len(balanced_poc))
print("Expected: 21 clients and 2,520 pages")
print("Signal rows:", len(signal_frame))
print("\nSTALENESS SIGNAL TABLE")
display(staleness_table)


# STEP 1B — audit CTR relative to March search position.
# Uses March-only aggregate CTR and March-only median position.
# April remains outcome/evaluation only.

march_ctr_position = con.sql(f"""
    WITH daily AS (
        SELECT
            f.client_hash_id,
            f.content_hash_id,
            f.gsc_clicks::DOUBLE AS clicks,
            f.gsc_impressions::DOUBLE AS impressions,
            CASE
                WHEN f.gsc_avg_position >= 1
                THEN f.gsc_avg_position::DOUBLE
                ELSE NULL
            END AS valid_position
        FROM {MARCH} AS f
        INNER JOIN balanced_keys AS k USING (client_hash_id, content_hash_id)
        WHERE f.gsc_data_available IS TRUE
    )
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(clicks) / NULLIF(SUM(impressions), 0) AS aggregate_ctr,
        MEDIAN(valid_position) AS median_position
    FROM daily
    GROUP BY client_hash_id, content_hash_id
""").df()

ctr_signal = signal_frame[
    ["client_hash_id", "content_hash_id", "future_impression_change", "future_decline"]
].merge(
    march_ctr_position,
    on=["client_hash_id", "content_hash_id"],
    how="inner",
)

ctr_signal["position_band"] = pd.cut(
    ctr_signal["median_position"],
    bins=[0, 3, 10, 20, float("inf")],
    labels=["1-3", "4-10", "11-20", "21+"],
    include_lowest=True,
)

# March-only percentile rank of CTR among pages with similar search position.
# method='average' handles ties (including zero CTR) without inventing ordering.
ctr_signal["ctr_within_position_percentile"] = (
    ctr_signal
    .groupby("position_band", observed=False)["aggregate_ctr"]
    .rank(method="average", pct=True)
)

ctr_signal["ctr_vs_position_bucket"] = pd.cut(
    ctr_signal["ctr_within_position_percentile"],
    bins=[0.0, 0.25, 0.50, 0.75, 1.0],
    labels=[
        "bottom 25%",
        "25-50%",
        "50-75%",
        "top 25%",
    ],
    include_lowest=True,
)

position_band_diagnostic = (
    ctr_signal
    .groupby("position_band", observed=False)
    .agg(
        n=("content_hash_id", "size"),
        median_position=("median_position", "median"),
        median_ctr=("aggregate_ctr", "median"),
        zero_ctr_rate=("aggregate_ctr", lambda s: float((s == 0).mean())),
    )
    .reset_index()
)
position_band_diagnostic["zero_ctr_rate_pct"] = (
    100 * position_band_diagnostic.pop("zero_ctr_rate")
)

ctr_position_table = (
    ctr_signal
    .groupby("ctr_vs_position_bucket", observed=False)
    .agg(
        n=("content_hash_id", "size"),
        decline_rate=("future_decline", "mean"),
        median_future_impression_change=("future_impression_change", "median"),
        mean_future_impression_change=("future_impression_change", "mean"),
        median_within_position_percentile=(
            "ctr_within_position_percentile", "median"
        ),
    )
    .reset_index()
)

ctr_position_table["decline_rate_pct"] = (
    100 * ctr_position_table.pop("decline_rate")
)

print("\nPOSITION-BAND DIAGNOSTIC (MARCH ONLY)")
display(position_band_diagnostic)

print("\nCTR-VS-POSITION SIGNAL TABLE")
display(ctr_position_table)


Locked POC clients: 21
Locked POC pages: 2520
Expected: 21 clients and 2,520 pages
Signal rows: 2520

STALENESS SIGNAL TABLE


,age_bucket,n,median_future_impression_change,mean_future_impression_change,decline_rate_pct
0,0-90 days,1023,-0.085986,0.262161,55.718475
1,91-180 days,265,-0.305471,-0.128728,72.075472
2,181-365 days,740,-0.224764,0.018090,65.810811
3,366+ days,492,-0.497090,-0.385274,84.959350



POSITION-BAND DIAGNOSTIC (MARCH ONLY)


,position_band,n,median_position,median_ctr,zero_ctr_rate_pct
0,1-3,159,2.323664,0.003486,18.238994
1,4-10,1545,6.223301,0.001688,29.514563
2,11-20,405,13.500000,0.000719,42.469136
3,21+,411,32.119048,0.000000,63.746959



CTR-VS-POSITION SIGNAL TABLE


,ctr_vs_position_bucket,n,median_future_impression_change,mean_future_impression_change,median_within_position_percentile,decline_rate_pct
0,bottom 25%,667,-0.346976,-0.197757,0.147896,74.662669
1,25-50%,648,-0.301211,-0.133100,0.327276,71.141975
2,50-75%,582,-0.252960,0.088778,0.637864,65.463918
3,top 25%,623,-0.047109,0.360181,0.877023,52.327448


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.